In [1]:
# data cleaning
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

df = pd.read_excel("PCOS_data_without_infertility.xlsx", sheet_name="Full_new")

# drop empty trailing columns
df = df.drop(columns=[c for c in df.columns if str(c).startswith("Unnamed")])

# strip whitespace from headers 
df.columns = [c.strip() for c in df.columns]

# auto-coerce columns for consistent data types
for c in df.columns:
    if df[c].dtype == "object":
        coerced = pd.to_numeric(df[c], errors="coerce")
        if coerced.notna().mean() > 0.8:
            df[c] = coerced

TARGET = "PCOS (Y/N)"
print("Data Frame:", df.shape)

Data Frame: (541, 44)


In [6]:
# how many women have PCOS & how many don't 
vc = df[TARGET].value_counts().sort_index()
print(f"No PCOS: {vc.get(0,0)}   PCOS: {vc.get(1,0)}")

No PCOS: 364   PCOS: 177


In [9]:
#feature grouping
questions_data = ["Age (yrs)", "BMI", "Cycle(R/I)", "Cycle length(days)", "Weight gain(Y/N)",
                 "hair growth(Y/N)", "Skin darkening (Y/N)", "Hair loss(Y/N)", "Pimples(Y/N)",
                 "Fast food (Y/N)", "Reg.Exercise(Y/N)"]

clinical_data = ["Follicle No. (L)", "Follicle No. (R)", "Avg. F size (L) (mm)", "Avg. F size (R) (mm)",
            "Endometrium (mm)", "AMH(ng/mL)", "LH(mIU/mL)", "FSH(mIU/mL)", "FSH/LH",
            "TSH (mIU/L)", "PRL(ng/mL)"]

print(f"{len(questions_data)} questionnaire data: {len(clinical_data)} clinical/diagnostic data:")
missing = [c for c in questions_data + clinical_data if c not in df.columns]
print("Name mismatches:", missing)

11 questionnaire data: 11 clinical/diagnostic data:
Name mismatches: []


In [17]:
groups = {
 "(hair/acne/skin)": ["hair growth(Y/N)","Pimples(Y/N)","Skin darkening (Y/N)","Hair loss(Y/N)"],
 "Cycle / ovulation":                 ["Cycle(R/I)","Cycle length(days)"],
 "Metabolic / lifestyle":             ["BMI","Weight gain(Y/N)","Fast food (Y/N)","Reg.Exercise(Y/N)","Age (yrs)"],
 "Basic questionnaire":                 questions_data,
 "+ clinical": questions_data + clinical_data,
}
for name, cols in groups.items():
    auc = cross_val_score(build_pipe(), df[cols], y, cv=cv, scoring="roc_auc").mean()
    print(f"{auc:.3f}   {name}")

0.837   (hair/acne/skin)
0.738   Cycle / ovulation
0.807   Metabolic / lifestyle
0.883   Basic questionnaire
0.954   + clinical


In [38]:
# testing diff. models 
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

for name, clf in [("Logistic regression", LogisticRegression(max_iter=2000)),
                  ("Random forest", RandomForestClassifier(n_estimators=300, random_state=1)),
                  ("Gradient boosting", GradientBoostingClassifier(random_state=1))]:
    pipe = Pipeline([("impute", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler()), ("clf", clf)])
    s = cross_val_score(pipe, df[FINAL_FEATURES], y, cv=cv, scoring="roc_auc")
    print(f"{s.mean():.3f} ± {s.std():.3f}   {name}")

0.886 ± 0.019   Logistic regression
0.880 ± 0.016   Random forest
0.892 ± 0.022   Gradient boosting


In [12]:
# Understanding contribution of features 
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.inspection import permutation_importance

def build_pipe():
    return Pipeline([("impute", SimpleImputer(strategy="median")),
                     ("scale",  StandardScaler()),
                     ("lr",     LogisticRegression(max_iter=2000))])

X, y = df[questions_data], df[TARGET]
cv = StratifiedKFold(5, shuffle=True, random_state=1)

#PERMUTATION IMP
imp = np.zeros((cv.get_n_splits(), len(questions_data)))
for i, (tr, te) in enumerate(cv.split(X, y)):
    pipe = build_pipe().fit(X.iloc[tr], y.iloc[tr])
    r = permutation_importance(pipe, X.iloc[te], y.iloc[te],
                               scoring="roc_auc", n_repeats=20, random_state=1)
    imp[i] = r.importances_mean
perm = pd.Series(imp.mean(0), index=questions_data).sort_values(ascending=False)

#coef
coef = pd.Series(build_pipe().fit(X, y).named_steps["lr"].coef_[0], index=questions_data)

print(pd.DataFrame({"perm_importance (AUC drop)": perm.round(4),
                    "direction (coef)": coef.reindex(perm.index).round(3)}))

                      perm_importance (AUC drop)  direction (coef)
hair growth(Y/N)                          0.0381             0.629
Skin darkening (Y/N)                      0.0335             0.605
Cycle(R/I)                                0.0288             0.562
Weight gain(Y/N)                          0.0265             0.578
Pimples(Y/N)                              0.0108             0.428
Age (yrs)                                 0.0090            -0.372
Fast food (Y/N)                           0.0086             0.341
Cycle length(days)                        0.0034            -0.223
Reg.Exercise(Y/N)                        -0.0001             0.142
Hair loss(Y/N)                           -0.0007            -0.077
BMI                                      -0.0009            -0.100


In [14]:
#finalizing feature group 
sets = {
 "Full 11":        questions_data,
 "Top 8":         [c for c in questions_data if c not in
                      ["BMI", "Hair loss(Y/N)", "Reg.Exercise(Y/N)"]],
 "Top 4": ["hair growth(Y/N)", "Skin darkening (Y/N)",
                        "Cycle(R/I)", "Weight gain(Y/N)"],
}
for name, cols in sets.items():
    s = cross_val_score(build_pipe(), df[cols], y, cv=cv, scoring="roc_auc")
    print(f"{s.mean():.3f} ± {s.std():.3f}   {name}")

0.883 ± 0.016   Full 11
0.886 ± 0.019   Top 8
0.862 ± 0.021   Top 4


In [27]:
# Model fitting 

FINAL_FEATURES = [
    "hair growth(Y/N)",       
    "Skin darkening (Y/N)",   
    "Cycle(R/I)",            
    "Weight gain(Y/N)",       
    "Pimples(Y/N)",           
    "Age (yrs)",              
    "Fast food (Y/N)",        
    "Cycle length(days)",     
]

def build_pipe():
    return Pipeline([("impute", SimpleImputer(strategy="median")),
                     ("scale",  StandardScaler()),
                     ("lr",     LogisticRegression(max_iter=2000))])

X, y = df[FINAL_FEATURES], df[TARGET]
cv = StratifiedKFold(5, shuffle=True, random_state=1)


oof = cross_val_predict(build_pipe(), X, y, cv=cv, method="predict_proba")[:, 1]
print(f"Out-of-fold AUC: {roc_auc_score(y, oof):.3f}")  

model = build_pipe().fit(X, y)   # final model trained on all rows

Out-of-fold AUC: 0.885


In [34]:
#response training and model prediction

LABELS = {
    "hair growth(Y/N)": "excess hair growth",
    "Skin darkening (Y/N)": "skin darkening",
    "Cycle(R/I)": "irregular cycles",
    "Weight gain(Y/N)": "weight gain",
    "Pimples(Y/N)": "acne",
    "Age (yrs)": "age",
    "Fast food (Y/N)": "diet",
    "Cycle length(days)": "cycle length",
}

def predict_risk(answers: dict):
    x = pd.DataFrame([{f: answers.get(f, np.nan) for f in FINAL_FEATURES}])
    p = float(model.predict_proba(x)[0, 1])

    xs = model.named_steps["scale"].transform(model.named_steps["impute"].transform(x))[0]
    contrib = pd.Series(xs * model.named_steps["lr"].coef_[0], index=FINAL_FEATURES)
    top = [LABELS[f] for f in contrib[contrib > 0].sort_values(ascending=False).head(3).index]

    band = risk_band(p)
    msg = {
        "Low":      "Your answers suggest a lower likelihood of PCOS. Keep tracking your cycle; see a clinician if symptoms change.",
        "Moderate": "Some of your answers are associated with PCOS. Worth raising with a clinician at your next visit.",
        "High":     "Several answers are commonly associated with PCOS. Consider booking a clinical evaluation to check.",
    }[band]
    if band == "Low":
        top = []
    return {"band": band, "score": round(p, 2), "top_factors": top, "message": msg,
            "Disclaimer": "Educational screening only for awareness only, for clinical diagnosis, please consult a clinican"}

#test 
for label, a in {
 "many symptoms": {"hair growth(Y/N)":1,"Skin darkening (Y/N)":1,"Cycle(R/I)":4,"Weight gain(Y/N)":1,
                   "Pimples(Y/N)":1,"Age (yrs)":24,"Fast food (Y/N)":1,"Cycle length(days)":2},
 "few symptoms":  {"hair growth(Y/N)":0,"Skin darkening (Y/N)":0,"Cycle(R/I)":2,"Weight gain(Y/N)":0,
                   "Pimples(Y/N)":0,"Age (yrs)":30,"Fast food (Y/N)":0,"Cycle length(days)":5},
 "borderline":    {"hair growth(Y/N)":0,"Skin darkening (Y/N)":0,"Cycle(R/I)":4,"Weight gain(Y/N)":1,
                   "Pimples(Y/N)":1,"Age (yrs)":27,"Fast food (Y/N)":1,"Cycle length(days)":4},
}.items():
    print(label, ":", predict_risk(a))

many symptoms : {'band': 'High', 'score': 0.98, 'top_factors': ['excess hair growth', 'skin darkening', 'irregular cycles'], 'message': 'Several answers are commonly associated with PCOS. Consider booking a clinical evaluation to check.', 'Disclaimer': 'Educational screening only for awareness only, for clinical diagnosis, please consult a clinican'}
few symptoms : {'band': 'Low', 'score': 0.03, 'top_factors': [], 'message': 'Your answers suggest a lower likelihood of PCOS. Keep tracking your cycle; see a clinician if symptoms change.', 'Disclaimer': 'Educational screening only for awareness only, for clinical diagnosis, please consult a clinican'}
borderline : {'band': 'High', 'score': 0.69, 'top_factors': ['irregular cycles', 'weight gain', 'acne'], 'message': 'Several answers are commonly associated with PCOS. Consider booking a clinical evaluation to check.', 'Disclaimer': 'Educational screening only for awareness only, for clinical diagnosis, please consult a clinican'}


In [36]:
#saving model 

import json

imp   = model.named_steps["impute"]
scal  = model.named_steps["scale"]
lr    = model.named_steps["lr"]

artifact = {
    "features":   FINAL_FEATURES,
    "labels":     LABELS,
    "medians":    imp.statistics_.tolist(),      # for missing answers
    "means":      scal.mean_.tolist(),
    "scales":     scal.scale_.tolist(),
    "coefficients": lr.coef_[0].tolist(),
    "intercept":  float(lr.intercept_[0]),
    "low_cut":    LOW_CUT,
    "high_cut":   HIGH_CUT,
    "notes": "Logistic regression on questionnaire features, Kottarathil PCOS dataset (Kerala, n=541). Screening only, not diagnostic.",
}
with open("pcos_model.json", "w") as f:
    json.dump(artifact, f, indent=2)
print("pcos_model.json")

pcos_model.json
